# Spoken-English Grammar Scoring
**SHL Hiring Assessment 2026**

The task is to predict a grammar score from 0 to 5 for each spoken-English audio clip.

## Approach
| Stage | What happens |
|---|---|
| **1. Speech-to-text** | Whisper `large-v3-turbo` transcribes each clip verbatim. Filler words, repetitions and false starts are kept, because they are part of the grammar signal. |
| **2. Text regression model** | `microsoft/deberta-v3-large` with LoRA adapters and a single-output regression head. Targets are standardised (z-scored) using each fold's training statistics. |
| **3. Validation** | 5-fold stratified cross-validation. Out-of-fold (OOF) predictions give an RMSE and Pearson correlation over every training clip. |
| **4. Inference** | The 5 fold models' test predictions are averaged, clipped to [0, 5] and rounded to one decimal. |

## Generated files (in `/kaggle/working`)
| File | Contents |
|---|---|
| `train_labels.csv`, `test_labels.csv` | Whisper transcripts with their labels |
| `deberta_lora_adapter_fold{k}/` | Fold *k* LoRA adapter, regression head and `norm.json` (the fold's label mean and std) |
| `oof_fold{k}.npy` | Fold *k* validation predictions. Also used to resume an interrupted run. |
| `oof_deberta.csv` | Out-of-fold predictions for all training clips |
| `submission.csv` | Final test predictions (`filename,label`) |

Running the notebook top to bottom reproduces all of the above. If a session is interrupted, re-running the training section skips folds whose adapter and OOF file already exist.

## 1. Environment setup

In [1]:
!pip install -U openai-whisper
!pip uninstall -y torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 15.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=cb2fdd4d2f64c9c34df170753e8be0031599596212f4db36ebcbe415ff32e188
  Stored in directory: /root/.cache/pip/wheels/ca/58/d5/fb4539ad74c3ca81eb40f7eda020ac77d080b33ad57449d485
Successfully built openai-whisper
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import glob
import json
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model
from scipy.stats import pearsonr
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {DEVICE} | GPUs visible: {torch.cuda.device_count()}")

torch 2.11.0+cu128 | device: cuda | GPUs visible: 1


## 2. Configuration
All paths, model choices and hyperparameters are defined in this one place.

In [3]:
@dataclass(frozen=True)
class Config:
    # ---- paths ----
    data_path: str = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/"
    work_dir: str = "/kaggle/working"

    # ---- speech-to-text ----
    whisper_model: str = "large-v3-turbo"
    whisper_prompt: str = (
        "Transcribe the audio verbatim, word for word. "
        "Preserve all spoken words, repetitions, filler words, "
        "false starts, stutters, and incomplete sentences. "
        "Do not paraphrase, summarize, correct grammar, or omit speech. "
        "Do not add words that are not audible."
    )

    # ---- text model ----
    model_name: str = "microsoft/deberta-v3-large"
    prompt_template: str = "Transcript of a spoken English answer:\n{}\n\nGrammar score:"
    max_length: int = 1024

    # ---- cross-validation ----
    n_folds: int = 5
    seed: int = 42

    # ---- LoRA ----
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.1
    lora_target_modules: tuple = ("query_proj", "key_proj", "value_proj", "dense")
    lora_modules_to_save: tuple = ("classifier", "pooler")

    # ---- training ----
    epochs: int = 5
    learning_rate: float = 1e-4
    train_batch_size: int = 1
    eval_batch_size: int = 4
    grad_accum_steps: int = 8          # effective batch size = 1 x 8
    warmup: float = 0.1                # passed as warmup_steps; a value < 1 is a fraction of total steps
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0

    # ---- inference ----
    predict_batch_size: int = 8
    score_min: float = 0.0
    score_max: float = 5.0
    round_decimals: int = 1

    # ---- output locations ----
    def path(self, *parts: str) -> str:
        return os.path.join(self.work_dir, *parts)

    def transcripts_path(self, mode: str) -> str:
        return self.path(f"{mode}_labels.csv")

    def adapter_dir(self, fold: int) -> str:
        return self.path(f"deberta_lora_adapter_fold{fold}")

    def oof_fold_path(self, fold: int) -> str:
        return self.path(f"oof_fold{fold}.npy")


CFG = Config()

## 3. Speech-to-text
Each clip is transcribed with Whisper and paired with its label from `<split>.csv`. The result is written to `<split>_labels.csv`, so later stages (and re-runs) read transcripts from disk.

In [4]:
class AudioTranscriber:
    """Transcribes one split's audio clips with Whisper and pairs each transcript with its label."""

    def __init__(self, cfg: Config, mode: str = "train"):
        self.cfg = cfg
        self.mode = mode

    def audio_files(self) -> list:
        return glob.glob(os.path.join(self.cfg.data_path, self.mode, "*.wav"))

    def labels(self) -> pd.DataFrame:
        return pd.read_csv(os.path.join(self.cfg.data_path, f"{self.mode}.csv"))

    def transcribe(self) -> dict:
        """Returns {filename: transcript}."""
        import whisper

        model = whisper.load_model(self.cfg.whisper_model)
        transcripts = {}
        for audio_path in tqdm(self.audio_files(), desc=f"Transcribing {self.mode}"):
            result = model.transcribe(
                audio_path,
                language="English",
                task="transcribe",
                temperature=0,
                condition_on_previous_text=False,
                initial_prompt=self.cfg.whisper_prompt,
            )
            transcripts[os.path.basename(audio_path)] = result["text"]

        del model
        gc.collect()
        torch.cuda.empty_cache()
        return transcripts

    def compose_dataset(self) -> pd.DataFrame:
        """Transcribes the split, attaches labels and saves it to `<mode>_labels.csv`."""
        transcripts = self.transcribe()
        labels = self.labels()
        rows = [
            {
                "filename": filename,
                "transcription": text,
                "label": labels.loc[labels["filename"] == filename, "label"].item(),
            }
            for filename, text in transcripts.items()
        ]
        df = pd.DataFrame(rows, columns=["filename", "transcription", "label"])
        df.to_csv(self.cfg.transcripts_path(self.mode), index=False)
        return df

In [5]:
AudioTranscriber(CFG, mode="train").compose_dataset()

100%|██████████████████████████████████████| 1.51G/1.51G [00:05<00:00, 308MiB/s]
Transcribing train: 100%|██████████| 769/769 [21:16<00:00,  1.66s/it]


,filename,transcription,label
0,audio_49.wav,"When I plan a vacation or a travel, I like to...",3.0
1,audio_663.wav,"Oh yes, I enjoy, I always enjoy playing tenni...",3.0
2,audio_5059.wav,"Alright, my favorite is that I like to visit....",0.0
3,audio_90.wav,The most enchanting schools always have large...,4.0
4,audio_581.wav,My goal is to become an electrical employee. ...,1.5
...,...,...,...
764,audio_473.wav,Recently I visited Shanai city with one of fr...,2.0
765,audio_651.wav,I'm inside of a very crowded market in the ce...,5.0
766,audio_724.wav,The following is a scene of a school playgrou...,5.0
767,audio_490.wav,My favorite hobby is to irritate my best frie...,2.5


## 4. Data preparation
- Each transcript is wrapped in a short prompt template.
- Clips are assigned to 5 cross-validation folds, stratified on the rounded score so that every fold has a similar label distribution.
- Labels are standardised with the training-fold mean and std, so no statistics leak from a fold's validation clips.

In [6]:
def load_tokenizer(cfg: Config):
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
    tokenizer.truncation_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def assign_folds(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Adds a `fold` column (0..n_folds-1), stratified on the rounded label."""
    df = df.reset_index(drop=True)
    df["fold"] = -1
    skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)
    for fold, (_, val_idx) in enumerate(skf.split(df, df["label"].round().astype(int))):
        df.loc[val_idx, "fold"] = fold
    return df


def load_training_data(cfg: Config) -> pd.DataFrame:
    df = pd.read_csv(cfg.transcripts_path("train"))
    df["prompt"] = df["transcription"].map(cfg.prompt_template.format)
    return assign_folds(df, cfg)


def to_dataset(frame: pd.DataFrame, mean: float, std: float, tokenizer, cfg: Config) -> Dataset:
    """Tokenised dataset with standardised regression targets."""
    ds = Dataset.from_dict({
        "prompt": frame["prompt"].to_list(),
        "labels": ((frame["label"] - mean) / std).astype("float32").to_list(),
    })
    return ds.map(
        lambda batch: tokenizer(batch["prompt"], truncation=True, max_length=cfg.max_length),
        batched=True,
        remove_columns=["prompt"],
    )

In [7]:
tokenizer = load_tokenizer(CFG)
train_data = load_training_data(CFG)
print(train_data.groupby("fold")["label"].agg(["size", "mean"]))

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

      size      mean
fold                
0      154  3.318182
1      154  3.295455
2      154  3.314935
3      154  3.305195
4      153  3.333333


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


## 5. Model
DeBERTa-v3-large with a single-output regression head. LoRA adapters go on the attention and dense projections. The classifier and pooler are trained in full. Gradient checkpointing keeps memory within a single 16 GB GPU.

In [8]:
def make_lora_config(cfg: Config) -> LoraConfig:
    return LoraConfig(
        task_type="SEQ_CLS",
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias="none",
        target_modules=list(cfg.lora_target_modules),
        modules_to_save=list(cfg.lora_modules_to_save),
    )


def build_model(cfg: Config, tokenizer):
    """A fresh base model with LoRA adapters attached; called once per fold."""
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name, num_labels=1, problem_type="regression", dtype=torch.float32
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()

    model = get_peft_model(model, make_lora_config(cfg))
    model.print_trainable_parameters()
    return model

## 6. Training with 5-fold cross-validation
Each fold trains a fresh model, saves its adapter and label statistics, and records predictions for its validation clips. Together those predictions cover every training clip once, giving the out-of-fold CV score.

In [9]:
class RegressionTrainer(Trainer):
    """Computes MSE in fp32 so that mixed-precision (fp16) logits don't clash with fp32 labels."""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        preds = outputs.logits.squeeze(-1).float()
        loss = F.mse_loss(preds, labels.float())
        return (loss, outputs) if return_outputs else loss


def make_compute_metrics(mean: float, std: float):
    """RMSE and Pearson on the original score scale, using the fold's label statistics."""
    def compute_metrics(eval_pred):
        preds = eval_pred.predictions.squeeze(-1) * std + mean
        labels = eval_pred.label_ids * std + mean
        return {
            "rmse": float(np.sqrt(np.mean((preds - labels) ** 2))),
            "pearson": float(pearsonr(preds, labels)[0]),
        }
    return compute_metrics


def make_training_args(cfg: Config) -> TrainingArguments:
    use_bf16 = torch.cuda.is_bf16_supported()
    return TrainingArguments(
        output_dir=cfg.path("deberta_lora_grammar"),
        num_train_epochs=cfg.epochs,
        learning_rate=cfg.learning_rate,
        per_device_train_batch_size=cfg.train_batch_size,
        per_device_eval_batch_size=cfg.eval_batch_size,
        gradient_accumulation_steps=cfg.grad_accum_steps,
        warmup_steps=cfg.warmup,
        lr_scheduler_type="cosine",
        weight_decay=cfg.weight_decay,
        max_grad_norm=cfg.max_grad_norm,
        bf16=use_bf16,
        fp16=not use_bf16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=10,
        report_to="none",
        label_names=["labels"],
        seed=cfg.seed,
    )


def rmse(pred, target) -> float:
    return float(np.sqrt(np.mean((np.asarray(pred) - np.asarray(target)) ** 2)))


def train_fold(df: pd.DataFrame, fold: int, tokenizer, training_args: TrainingArguments, cfg: Config) -> np.ndarray:
    """Trains one fold and returns its validation predictions (original score scale).

    Saves the fold's adapter, its label mean/std (`norm.json`) and its validation
    predictions. If those already exist, the fold is skipped and the saved
    predictions are returned, which lets an interrupted run resume.
    """
    adapter_dir, oof_path = cfg.adapter_dir(fold), cfg.oof_fold_path(fold)
    if os.path.exists(os.path.join(adapter_dir, "norm.json")) and os.path.exists(oof_path):
        print(f"fold {fold}: already trained, skipping")
        return np.load(oof_path)

    train_df, val_df = df[df.fold != fold], df[df.fold == fold]
    mean, std = float(train_df["label"].mean()), float(train_df["label"].std())

    train_ds = to_dataset(train_df, mean, std, tokenizer, cfg)
    val_ds = to_dataset(val_df, mean, std, tokenizer, cfg)
    print(f"===== fold {fold}: train {len(train_ds)} / val {len(val_ds)} =====")

    model = build_model(cfg, tokenizer)
    trainer = RegressionTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=make_compute_metrics(mean, std),
    )
    trainer.model_accepts_loss_kwargs = False   # Trainer averages the loss over the accumulation steps
    trainer.train()

    val_pred = np.clip(
        trainer.predict(val_ds).predictions.squeeze(-1) * std + mean, cfg.score_min, cfg.score_max
    )
    np.save(oof_path, val_pred)
    model.save_pretrained(adapter_dir)          # LoRA adapters + regression head only (a few MB)
    with open(os.path.join(adapter_dir, "norm.json"), "w") as f:
        json.dump({"mean": mean, "std": std}, f)
    print(f"fold {fold} RMSE: {rmse(val_pred, val_df['label']):.4f}")

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    return val_pred


def run_cross_validation(df: pd.DataFrame, tokenizer, cfg: Config) -> pd.DataFrame:
    """Trains every fold, reports the out-of-fold CV score and saves `oof_deberta.csv`."""
    training_args = make_training_args(cfg)
    oof = np.zeros(len(df))
    for fold in range(cfg.n_folds):
        oof[df.index[df.fold == fold]] = train_fold(df, fold, tokenizer, training_args, cfg)

    df = df.copy()
    df["oof"] = oof
    print(f"CV RMSE: {rmse(df['oof'], df['label']):.4f} | Pearson: {pearsonr(df['oof'], df['label'])[0]:.4f}")
    df[["filename", "label", "fold", "oof"]].to_csv(cfg.path("oof_deberta.csv"), index=False)
    return df

In [10]:
cv_results = run_cross_validation(train_data, tokenizer, CFG)

Map:   0%|          | 0/615 [00:00<?, ? examples/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

===== fold 0: train 615 / val 154 =====


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  874MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.de

trainable params: 8,128,513 || all params: 443,191,298 || trainable%: 1.8341


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Rmse,Pearson
1,0.709430,0.503663,0.876649,0.765395
2,0.401317,0.357581,0.738657,0.826017
3,0.414991,0.437589,0.817126,0.836746
4,0.333701,0.355723,0.736735,0.840074
5,0.313425,0.309009,0.686659,0.839022


fold 4 RMSE: 0.6866
CV RMSE: 0.7255 | Pearson: 0.8114


## 7. Inference
The test clips are transcribed with the same Whisper setup. Each fold model predicts every test clip, using its own label mean and std to convert back to the score scale. The 5 predictions are then averaged.

In [11]:
def load_fold_model(fold: int, tokenizer, cfg: Config):
    """Loads a fold's adapter merged into the base model. Returns (model, label mean, label std)."""
    adapter_dir = cfg.adapter_dir(fold)
    with open(os.path.join(adapter_dir, "norm.json")) as f:
        norm = json.load(f)

    base = AutoModelForSequenceClassification.from_pretrained(cfg.model_name, num_labels=1, dtype=torch.float16)
    base.config.pad_token_id = tokenizer.pad_token_id
    model = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload().to(DEVICE).eval()
    return model, norm["mean"], norm["std"]


@torch.no_grad()
def predict(model, texts: list, mean: float, std: float, tokenizer, cfg: Config) -> np.ndarray:
    """Scores raw transcripts with one model; returns predictions clipped to the score range."""
    out = []
    for i in range(0, len(texts), cfg.predict_batch_size):
        batch = [cfg.prompt_template.format(t) for t in texts[i:i + cfg.predict_batch_size]]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=cfg.max_length,
                        return_tensors="pt").to(DEVICE)
        out.append(model(**enc).logits.squeeze(-1).float().cpu().numpy())
    z = np.concatenate(out)
    return np.clip(z * std + mean, cfg.score_min, cfg.score_max)


def predict_ensemble(texts: list, tokenizer, cfg: Config) -> np.ndarray:
    """Averages the predictions of all fold models."""
    fold_preds = []
    for fold in range(cfg.n_folds):
        model, mean, std = load_fold_model(fold, tokenizer, cfg)
        fold_preds.append(predict(model, texts, mean, std, tokenizer, cfg))
        del model
        gc.collect()
        torch.cuda.empty_cache()

    fold_preds = np.array(fold_preds)
    print("avg disagreement between fold models:", fold_preds.std(axis=0).mean().round(3))
    return np.round(fold_preds.mean(axis=0), cfg.round_decimals)

In [12]:
AudioTranscriber(CFG, mode="test").compose_dataset()

test_data = pd.read_csv(CFG.transcripts_path("test"))
test_data["label"] = predict_ensemble(test_data["transcription"].fillna("").tolist(), tokenizer, CFG)
print(test_data)

Transcribing test: 100%|██████████| 216/216 [05:37<00:00,  1.56s/it]


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.de

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.de

...

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.de

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.de

avg disagreement between fold models: 0.1
          filename                                      transcription  label
0     audio_49.wav   My favourite place to visit is at my nan's. T...    4.7
1     audio_90.wav   Well, the floods are very natural disasters t...    3.3
2     audio_77.wav   Hi. My goal in life is to have a good work-li...    4.9
3     audio_66.wav   What is the word? What is the word? What is t...    0.6
4     audio_54.wav   My favorite place to visit is Srinagar becaus...    3.1
..             ...                                                ...    ...
211   audio_74.wav   Best day of my life. It was probably more of ...    4.8
212   audio_51.wav   very happy to see at the TV okay my whole fam...    2.0
213  audio_163.wav   My favorite fictional character is Harry Moon...    3.2
214   audio_91.wav   A flood is a terrifying scene. Sometimes men ...    3.5
215  audio_151.wav   Basketball is my favorite sport. I like how i...    3.7

[216 rows x 3 columns]


## 8. Submission

In [13]:
def write_submission(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    submission = df[["filename", "label"]]
    submission.to_csv(cfg.path("submission.csv"), index=False)
    return submission


submission = write_submission(test_data, CFG)
print(submission.shape)
print(submission.head())
print(submission.describe())

(216, 2)
       filename  label
0  audio_49.wav    4.7
1  audio_90.wav    3.3
2  audio_77.wav    4.9
3  audio_66.wav    0.6
4  audio_54.wav    3.1
            label
count  216.000000
mean     3.245833
std      0.921213
min      0.600000
25%      2.500000
50%      3.200000
75%      3.800000
max      5.000000
